# 📘 KỸ THUẬT LẬP TRÌNH C++ — CHƯƠNG IX & X
## Kế Thừa (Inheritance) & Đa Hình (Polymorphism)

> **Môn:** Kỹ thuật Lập trình C++ | **Chuẩn:** C++17
> Mỗi ví dụ = **Cell 1** ghi file `.cpp` + **Cell 2** biên dịch & chạy.

---
## 📋 MỤC LỤC

| Phần | Nội dung |
|:----:|----------|
| **IX.1–3** | Khái niệm, Lớp cơ sở, Lớp dẫn xuất |
| **IX.4–6** | Public / Protected / Private Inheritance |
| **IX.7** | Constructor trong kế thừa |
| **IX.8** | Destructor & Virtual Destructor ⚠️ |
| **IX.9** | Ghi đè hàm (override, final) |
| **X.1–3** | Khái niệm, Đa hình Tĩnh, Đa hình Động |
| **X.4** | Overloading vs Overriding |
| **X.5** | Hàm ảo, Hàm thuần ảo, Lớp trừu tượng |
| **X.6** | Bảng hàm ảo (vtable) — cơ chế bên trong |
| **X.7** | Con trỏ lớp cơ sở & Object Slicing |
| **X.8–9** | Liên kết Động & Liên kết Tĩnh |
| **★** | Bài tổng hợp: Hệ thống Quản lý Nhân Viên |


## ⚙️ SETUP — Cấu hình C++ trên Google Colab
```
Cell %%writefile → ghi code vào file .cpp
Cell !g++        → biên dịch và chạy ngay
```


In [ ]:
!g++ --version | head -1
print("✅ Sẵn sàng chạy C++ trên Colab!")

---
# IX. KẾ THỪA (INHERITANCE)

> Kế thừa cho phép lớp con **tái sử dụng** code từ lớp cha,
> **mở rộng** hành vi mà không cần viết lại từ đầu.

```
                    DongVat (Base Class)
                   /                    \
          Cho (Derived)              Meo (Derived)
               /
        ChoCon (Derived of Derived — đa cấp)
```

**Bốn lợi ích cốt lõi:**

| Lợi ích | Ý nghĩa |
|---------|---------|
| **Code Reuse** | Không lặp lại code chung |
| **IS-A Relationship** | Cho IS-A DongVat — rõ ràng về mặt khái niệm |
| **Extensibility** | Thêm loại mới mà không sửa code cũ |
| **Polymorphism** | Xử lý nhiều loại qua cùng 1 interface |


## IX.1–3 — Khái niệm · Lớp Cơ Sở · Lớp Dẫn Xuất

### 🧠 Ba khái niệm cốt lõi

**Lớp cơ sở (Base Class):**
- Lớp cha, tổng quát hơn — cung cấp interface và implementation chung
- Thường có virtual functions để cho phép override

**Lớp dẫn xuất (Derived Class):**
- Lớp con, cụ thể hơn — kế thừa tất cả non-private members của base
- Có thể **thêm** members mới và **override** methods

**Cú pháp:**
```cpp
class LopCon : [access] LopCha {
    // members bổ sung
};
```

### 📐 Quyền truy cập trong kế thừa
```
private   → CHỈ trong Base — Derived KHÔNG truy cập được
protected → Trong Base VÀ tất cả Derived classes
public    → Accessible ở mọi nơi
```

### 💡 Gọi phiên bản base từ override
```cpp
void Meo::gioi_thieu() override {
    DongVat::gioi_thieu();  // gọi cha trước
    cout << "Tôi là mèo!\n";
}
```

### 🔑 Kế thừa đa cấp
```
DongVat → Cho → ChoCon
ChoCon kế thừa TẤT CẢ từ cả Cho VÀ DongVat
```


In [ ]:
%%writefile ix13_basic.cpp
#include <iostream>
#include <string>
using namespace std;

// ─── LỚP CƠ SỞ ───────────────────────────────────────────────────────────────
class DongVat {
    string ten;      // private: CHỈ DongVat truy cập được
protected:
    int tuoi;        // protected: Derived classes cũng truy cập được!
public:
    DongVat(string t, int a) : ten(t), tuoi(a) {}
    virtual ~DongVat() {}

    string getTen() const { return ten; }
    void   an()     const { cout << ten << " dang an.\n"; }  // plain method

    virtual void keu()          const { cout << ten << " keu: ...\n"; }
    virtual void gioi_thieu()   const {
        cout << "Toi la " << ten << ", " << tuoi << " tuoi.\n";
    }
};

// ─── LỚP DẪN XUẤT 1: Cho ─────────────────────────────────────────────────────
class Cho : public DongVat {
    string giong;   // data member MỚI — chỉ có ở Cho
public:
    Cho(string t, int a, string g) : DongVat(t, a), giong(g) {}

    void keu() const override {   // override — thay đổi hành vi
        cout << getTen() << " sua: GAU GAU!\n";
    }
    void lay_do() const {         // method MỚI — không có trong DongVat
        cout << getTen() << " lay do vat!\n";
    }
    void kiem_tra() const {
        // Truy cập 'tuoi' (protected của DongVat) trực tiếp!
        cout << getTen() << " (" << tuoi << " tuoi) - " << giong << "\n";
    }
};

// ─── LỚP DẪN XUẤT 2: Meo ─────────────────────────────────────────────────────
class Meo : public DongVat {
public:
    Meo(string t, int a) : DongVat(t, a) {}

    void keu() const override {
        cout << getTen() << " keu: MEO MEO!\n";
    }
    void gioi_thieu() const override {  // override VÀ gọi base
        DongVat::gioi_thieu();           // <-- gọi phiên bản cha trước
        cout << "  (Toi la meo, thich ngu!)\n";
    }
};

// ─── KẾ THỪA ĐA CẤP: ChoCon kế thừa từ Cho ──────────────────────────────────
class ChoCon : public Cho {
public:
    ChoCon(string t) : Cho(t, 0, "Mixed") {}

    void keu() const override {
        cout << getTen() << " sua nho: gau!\n";
    }
};

int main() {
    DongVat dv("Thu khong ten", 5);
    Cho     cho("Rex", 3, "Becgie");
    Meo     meo("Mimi", 2);
    ChoCon  cc("Lucky");

    cout << "=== an() ke thua — goi cua DongVat ===\n";
    dv.an();   cho.an();   meo.an();   cc.an();

    cout << "\n=== keu() override ===\n";
    dv.keu();   // DongVat::keu
    cho.keu();  // Cho::keu
    meo.keu();  // Meo::keu
    cc.keu();   // ChoCon::keu

    cout << "\n=== gioi_thieu — goi ca base ===\n";
    meo.gioi_thieu();   // gọi DongVat::gioi_thieu() rồi thêm text

    cout << "\n=== Method rieng cua Cho (ke thua xuong ChoCon) ===\n";
    cho.lay_do();
    cho.kiem_tra();   // dùng protected member 'tuoi'
    cc.lay_do();      // ChoCon ke thua lay_do() tu Cho!

    cout << "\n=== IS-A: Cho IS-A DongVat ===\n";
    DongVat* p = &cho;   // Base pointer tro Derived object
    p->keu();            // goi Cho::keu() nho virtual dispatch!

    return 0;
}


In [ ]:
!g++ -std=c++17 -o ix13_out ix13_basic.cpp && ./ix13_out

## IX.4–6 — Ba kiểu kế thừa: Public · Protected · Private

### 📋 Bảng quy đổi quyền truy cập

| Access trong Base | `public` | `protected` | `private` |
|:-----------------:|:--------:|:-----------:|:---------:|
| `public`          | `public` | `protected` | `private` |
| `protected`       | `protected` | `protected` | `private` |
| `private`         | ❌ N/A | ❌ N/A | ❌ N/A |

### 🎯 Khi nào dùng loại nào?

```
public    → IS-A relationship: Cho IS-A DongVat           (~95% trường hợp)
protected → IS-A nhưng sealed: không muốn bên ngoài biết
private   → IMPLEMENTED-IN-TERMS-OF: DanXuat dùng Base bên trong
            (thường thay bằng composition tốt hơn)
```

### ⚠️ Default nếu không viết
```cpp
class Foo : Base { }   // DEFAULT = PRIVATE inheritance!
struct Foo : Base { }  // DEFAULT = PUBLIC inheritance!
```

### 💡 Re-expose member với `using`
Khi dùng protected/private inheritance, có thể "bật lại" một member:
```cpp
class PrivateDerived : private Base {
public:
    using Base::public_method;  // re-expose ra ngoài
};
```


In [ ]:
%%writefile ix46_access.cpp
#include <iostream>
using namespace std;

class CoSo {
public:
    void show_pub()  { cout << "  [public method]\n"; }
    int  pub_val = 1;
protected:
    void show_pro()  { cout << "  [protected method]\n"; }
    int  pro_val = 2;
private:
    void show_pri()  { /* private - khong ke thua duoc */ }
};

// ─── PUBLIC: pub→pub, pro→pro ────────────────────────────────────────────────
class PublicChild : public CoSo {
public:
    void demo() {
        show_pub();  // OK — van la public
        show_pro();  // OK — van la protected (visible inside class)
        // show_pri(); // ERROR: private khong ke thua
    }
};

// ─── PROTECTED: pub→protected, pro→protected ────────────────────────────────
class ProtectedChild : protected CoSo {
public:
    void demo() {
        show_pub();  // OK — nhung da la protected inside
        show_pro();  // OK — van la protected
    }
};

// ─── PRIVATE: pub→private, pro→private ──────────────────────────────────────
class PrivateChild : private CoSo {
public:
    void demo() {
        show_pub();  // OK — nhung da la private inside
        show_pro();  // OK — nhung da la private inside
    }
    using CoSo::show_pub;  // Re-expose ra public!
};

// ─── Kế thừa từ PublicChild ──────────────────────────────────────────────────
class GrandChild : public PublicChild {
public:
    void demo() {
        show_pub();   // OK — ke thua qua PublicChild
        show_pro();   // OK — van la protected
        // show_pri(); // ERROR: private khong ke thua duoc bao gio
    }
};

int main() {
    cout << "=== PUBLIC Inheritance ===\n";
    PublicChild pc;
    pc.demo();
    pc.show_pub();               // OK: van la public
    cout << "  pub_val=" << pc.pub_val << "\n";  // OK: public
    // pc.show_pro();            // COMMENT: ERROR - protected

    cout << "\n=== PROTECTED Inheritance ===\n";
    ProtectedChild prc;
    prc.demo();
    // prc.show_pub();           // COMMENT: ERROR - da thanh protected
    cout << "  (show_pub() an tu ben ngoai)\n";

    cout << "\n=== PRIVATE Inheritance ===\n";
    PrivateChild pvc;
    pvc.demo();
    pvc.show_pub();              // OK: vi re-expose bang 'using'
    // pvc.pub_val = 5;          // COMMENT: ERROR - da la private
    cout << "  (chi show_pub() duoc dung ngoai nho 'using')\n";

    cout << "\n=== Ke thua tiem tiep (GrandChild) ===\n";
    GrandChild gc;
    gc.demo();
    gc.show_pub();               // OK: public chain
    cout << "  pub_val=" << gc.pub_val << "\n";

    cout << "\n=== Ket luan ===\n";
    cout << "public:    IS-A rõ rang - pho bien nhat\n";
    cout << "protected: IS-A nhung sealed - it dung\n";
    cout << "private:   implemented-in-terms-of - hiem\n";
    return 0;
}


In [ ]:
!g++ -std=c++17 -o ix46_out ix46_access.cpp && ./ix46_out

## IX.7 — Constructor trong Kế thừa

### 🧠 Thứ tự khởi tạo (Construction Order)

```
LUÔN LUÔN: BASE khởi tạo TRƯỚC → DERIVED sau

Ví dụ 3 cấp: A → B → C
  Tạo C(args):  A() ctor  →  B() ctor  →  C() ctor
  Huỷ C:        ~C() dtor → ~B() dtor → ~A() dtor  (NGƯỢC LẠI!)
```

### 📐 Gọi Base Constructor từ Derived

```cpp
class Derived : public Base {
public:
    Derived(int x, int y) : Base(x), my_val(y) {  // gọi Base(x)
        // code của Derived
    }
};
```

> ⚠️ Nếu không gọi tường minh → compiler gọi **default constructor** của Base
> → Nếu Base không có default ctor → **LỖI BIÊN DỊCH!**

### 💡 Quy tắc truyền tham số theo chuỗi

```
C(a,b,c) : B(a,b), valC(c) { }
B(a,b)   : A(a),   valB(b) { }
A(a)     { /* khởi tạo valA = a */ }
```

### 🔑 Member Initializer List — thứ tự ưu tiên
```cpp
// PHẢI dùng initializer list cho:
class Foo : public Base {
    const int x;      // const member
    int& ref;         // reference member
    Base other;       // member không có default ctor
public:
    Foo(int v) : Base(v), x(v), ref(something), other(v) {}
};
```


In [ ]:
%%writefile ix7_ctor.cpp
#include <iostream>
using namespace std;

class A {
    int valA;
public:
    A()      : valA(0)  { cout << "[A] Default ctor  (valA=0)\n"; }
    A(int x) : valA(x)  { cout << "[A] Param ctor    (valA=" << x << ")\n"; }
    ~A()                { cout << "[A] Destructor    (valA=" << valA << ")\n"; }
    int getA() const { return valA; }
};

class B : public A {
    int valB;
public:
    // Goi A(10) tuong minh qua initializer list
    B() : A(10), valB(0) {
        cout << "[B] Default ctor  (valB=0, da goi A(10))\n";
    }
    B(int x, int y) : A(x), valB(y) {
        cout << "[B] Param ctor    (valA=" << x << ", valB=" << y << ")\n";
    }
    ~B() { cout << "[B] Destructor    (valB=" << valB << ")\n"; }
    int getB() const { return valB; }
};

class C : public B {
    int valC;
public:
    C(int a, int b, int c) : B(a, b), valC(c) {
        cout << "[C] Param ctor    (valC=" << c << ")\n";
    }
    ~C() { cout << "[C] Destructor    (valC=" << valC << ")\n"; }
    void show() const {
        cout << "Gia tri: A=" << getA() << " B=" << getB() << " C=" << valC << "\n";
    }
};

int main() {
    cout << "── Tao B() ──\n";
    cout << "Thu tu: A → B\n";
    {
        B b;
        // Khi ra khoi scope → destructor goi NGUOC LAI: B → A
    }

    cout << "\n── Tao C(1, 2, 3) ──\n";
    cout << "Thu tu: A → B → C\n";
    {
        C c(1, 2, 3);
        c.show();
        cout << "Huy: C → B → A\n";
    }

    cout << "\n── Luu y: khong goi tuong minh → default ctor ──\n";
    cout << "Neu Base khong co default ctor → LOI bien dich!\n";
    return 0;
}


In [ ]:
!g++ -std=c++17 -o ix7_out ix7_ctor.cpp && ./ix7_out

## IX.8 — Destructor trong Kế thừa & Virtual Destructor

### 🧠 Thứ tự hủy
```
Ngược lại hoàn toàn với Constructor:
DERIVED hủy TRƯỚC → BASE sau
C dtor → B dtor → A dtor
```

### 🔥 Virtual Destructor — VẤN ĐỀ NGHIÊM TRỌNG NHẤT!

Khi dùng **base pointer** trỏ đến derived object và `delete`:

```
KHÔNG virtual dtor:          CÓ virtual dtor:
Base* p = new Derived();     Base* p = new Derived();
delete p;                    delete p;
  → chỉ gọi ~Base()            → ~Derived() TRƯỚC, ~Base() SAU
  → ~Derived() KHÔNG gọi!       ✅ Đúng thứ tự!
  → MEMORY LEAK!!!
```

### 📐 Quy tắc vàng — PHẢI thuộc lòng!

```cpp
// Nếu class có BẤT KỲ virtual function nào →
// PHẢI khai báo virtual destructor!
class Base {
public:
    virtual ~Base() {}   // ← virtual destructor
};
```

### 💡 Tóm tắt khi nào cần virtual destructor
- ✅ Class có virtual functions → **LUÔN** làm virtual dtor
- ✅ Class được thiết kế để làm base class → **LUÔN** làm virtual dtor
- ❌ Leaf class không có derived → không cần (nhưng không hại)
- ❌ Class không dùng polymorphism → không cần


In [ ]:
%%writefile ix8_virtual_dtor.cpp
#include <iostream>
using namespace std;

// ═══ CASE 1: KHONG co virtual dtor → NGUY HIEM! ═════════════════════════════
class Base_Bad {
public:
    Base_Bad()  { cout << "[Base_Bad]    Constructor\n"; }
    ~Base_Bad() { cout << "[Base_Bad]    Destructor\n"; }  // NOT virtual!
};
class Derived_Bad : public Base_Bad {
    int* data;
public:
    Derived_Bad()  : data(new int[1000]) {
        cout << "[Derived_Bad] Constructor  — 1000 ints CAP PHAT\n";
    }
    ~Derived_Bad() {
        delete[] data;
        cout << "[Derived_Bad] Destructor  — 1000 ints GIAI PHONG\n";
    }
};

// ═══ CASE 2: CO virtual dtor → AN TOAN ═══════════════════════════════════════
class Base_Good {
public:
    Base_Good()          { cout << "[Base_Good]    Constructor\n"; }
    virtual ~Base_Good() { cout << "[Base_Good]    Destructor\n"; }  // VIRTUAL!
};
class Derived_Good : public Base_Good {
    int* data;
public:
    Derived_Good()  : data(new int[1000]) {
        cout << "[Derived_Good] Constructor — 1000 ints CAP PHAT\n";
    }
    ~Derived_Good() {
        delete[] data;
        cout << "[Derived_Good] Destructor — 1000 ints GIAI PHONG\n";
    }
};

int main() {
    cout << "══════════════════════════════════════════\n";
    cout << "CASE 1: Base pointer, NON-virtual destructor\n";
    cout << "══════════════════════════════════════════\n";
    Base_Bad* p1 = new Derived_Bad();
    delete p1;
    // Chi goi ~Base_Bad() → ~Derived_Bad() KHONG GOI → MEMORY LEAK!
    cout << "==> Derived_Bad dtor KHONG duoc goi → MEMORY LEAK!\n";

    cout << "\n══════════════════════════════════════════\n";
    cout << "CASE 2: Base pointer, VIRTUAL destructor\n";
    cout << "══════════════════════════════════════════\n";
    Base_Good* p2 = new Derived_Good();
    delete p2;
    // Goi ~Derived_Good() truoc, sau do ~Base_Good() ✓
    cout << "==> Ca hai dtor deu duoc goi dung thu tu!\n";

    cout << "\n══════════════════════════════════════════\n";
    cout << "CASE 3: Stack object (khong qua pointer)\n";
    cout << "══════════════════════════════════════════\n";
    {
        Derived_Bad d;  // Stack allocation
    }  // ~Derived_Bad → ~Base_Bad → OK (stack goi dung du virtual hay khong)
    cout << "(Stack object luon goi dtor dung thu tu!)\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -o ix8_out ix8_virtual_dtor.cpp && ./ix8_out

## IX.9 — Ghi đè hàm (Method Overriding)

### 🧠 Overriding là gì?
Derived class định nghĩa lại hàm **cùng tên, cùng signature** với Base class.
Hành vi mới thay thế hành vi cũ khi gọi qua base pointer.

### 📐 Từ khóa quan trọng (C++11)

**`override` — NÊN dùng mọi lúc:**
```cpp
void method() override { ... }
```
Compiler kiểm tra: "Có thực sự override không?"
→ Bắt lỗi typo hoặc sai signature **ngay lúc biên dịch**.

**`final` trên method:**
```cpp
void method() override final { }  // không ai được override method này nữa
```

**`final` trên class:**
```cpp
class LeafClass final { };  // không thể kế thừa từ class này
```

### ⚠️ Lỗi âm thầm khi không dùng `override`
```cpp
class Base { virtual void method(int x) {} };
class Derived : public Base {
    void method(float x) {}  // BUG! Tưởng override nhưng là OVERLOAD!
    // Compiler không báo lỗi → silent bug khó tìm ra
    
    void method(int x) override {}  // OK: override đúng signature
};
```

### 💡 Gọi phiên bản base từ override
```cpp
void Derived::method() override {
    Base::method();   // Gọi base version trước/sau
    // Code bổ sung
}
```


In [ ]:
%%writefile ix9_overriding.cpp
#include <iostream>
using namespace std;

class HinhHoc {
public:
    virtual void ve()    const { cout << "Ve hinh tong quat\n"; }
    virtual void mo_ta() const { cout << "Day la mot hinh hinh hoc.\n"; }
    virtual ~HinhHoc() {}
};

class HinhTron : public HinhHoc {
    double r;
public:
    HinhTron(double r) : r(r) {}

    void ve() const override {           // override keyword!
        cout << "Ve hinh tron (r=" << r << ")\n";
    }
    void mo_ta() const override {
        HinhHoc::mo_ta();                // goi phien ban cha
        cout << "Day la hinh tron ban kinh " << r << "\n";
    }
};

class HinhChuNhat : public HinhHoc {
    double d, r;
public:
    HinhChuNhat(double d, double r) : d(d), r(r) {}
    void ve() const override {
        cout << "Ve HCN (" << d << "x" << r << ")\n";
    }
};

// final tren METHOD — khong override duoc nua
class HinhVuong : public HinhChuNhat {
public:
    HinhVuong(double canh) : HinhChuNhat(canh, canh) {}
    void ve() const override final {      // FINAL tren method!
        cout << "Ve hinh vuong (canh=" << "?)\n";
    }
};

// final tren CLASS — khong ke thua duoc
class LeafClass final : public HinhHoc {
public:
    void ve() const override {
        cout << "Ve LeafClass (class nay khong the ke thua!)\n";
    }
};

// ─── Demo: override bat loi chu ky sai ───────────────────────────────────────
class BaseDemo {
public:
    virtual void test(int x) {
        cout << "BaseDemo::test(int " << x << ")\n";
    }
};
class DerivedDemo : public BaseDemo {
public:
    // void test(double x) {}             // Neu viet nhu nay:
    // void test(double x) override {}    // COMPILE ERROR! <- bat loi ngay!
    void test(int x) override {           // Dung: override voi dung signature
        cout << "DerivedDemo::test(int " << x << ")\n";
    }
};

int main() {
    HinhTron    ht(5);
    HinhChuNhat hcn(3, 4);
    HinhVuong   hv(6);
    LeafClass   lc;

    cout << "=== ve() override ===\n";
    ht.ve();  hcn.ve();  hv.ve();  lc.ve();

    cout << "\n=== mo_ta() goi ca base ===\n";
    ht.mo_ta();

    cout << "\n=== override bat loi ===\n";
    BaseDemo    bd;   DerivedDemo dd;
    bd.test(10);   dd.test(20);

    // class CannotDerive : public LeafClass {}; // ERROR: LeafClass is final
    cout << "\n==> 'override' dam bao dung signature tai compile time\n";
    cout << "==> 'final' ngan override tiep / ke thua tiep\n";
    return 0;
}


In [ ]:
!g++ -std=c++17 -o ix9_out ix9_overriding.cpp && ./ix9_out

---
# X. ĐA HÌNH (POLYMORPHISM)

> **Polymorphism** = "Nhiều hình thức" — cùng một giao diện,
> hành vi khác nhau tùy đối tượng cụ thể lúc thực thi.

```
                 ┌──────────────────────────────────────┐
                 │           POLYMORPHISM               │
                 └──────────────┬───────────────────────┘
                                │
               ┌────────────────┴──────────────────┐
               │                                   │
       Static (Tĩnh)                        Dynamic (Động)
    Compile-time                             Run-time
    Function Overloading                     Virtual Functions
    Operator Overloading                     Base Class Pointer
    Templates                                vtable Dispatch
```

**Polymorphism là nền tảng của:**
- OOP Design Patterns (Strategy, Observer, Factory...)
- SOLID Principles (Open/Closed, Liskov Substitution)
- Code mở rộng không cần sửa đổi


## X.1–3 — Khái niệm · Đa hình Tĩnh · Đa hình Động

### X.1 — Khái niệm
Cùng một lệnh gọi `shape->draw()`, nhưng thực thi **khác nhau** tùy `shape` là `Circle` hay `Rectangle`.

### X.2 — Đa hình Tĩnh (Compile-time / Static Polymorphism)

| Cơ chế | Resolved khi | Ví dụ |
|--------|:------------:|-------|
| Function Overloading | Compile time | `tinh(int)`, `tinh(double)` |
| Operator Overloading | Compile time | `a + b` cho nhiều kiểu |
| Templates | Compile time | `sort<T>()`, `vector<T>` |

```cpp
void in(int x)    { cout << "[int] " << x; }
void in(double x) { cout << "[dbl] " << x; }
in(5);    // → in(int)    — quyết định tại COMPILE TIME
in(3.14); // → in(double) — quyết định tại COMPILE TIME
```

### X.3 — Đa hình Động (Run-time / Dynamic Polymorphism)

```cpp
HinhHoc* shapes[] = {new HinhTron(5), new HinhChuNhat(3,4)};
for (auto s : shapes)
    s->dien_tich();   // RUN TIME mới biết gọi HinhTron hay HinhChuNhat!
```

### ⚖️ So sánh

| | Static | Dynamic |
|--|:------:|:-------:|
| Thời điểm | Compile | Runtime |
| Cơ chế | Name mangling | vtable lookup |
| Tốc độ | ✅ Nhanh hơn | Chậm hơn 1 chút |
| Linh hoạt | Kém hơn | ✅ Cao hơn |
| Cần | - | `virtual` + pointer/reference |


## X.4 — Overloading vs Overriding — So sánh chi tiết

### 📋 Bảng so sánh toàn diện

| Tiêu chí | **Overloading** | **Overriding** |
|---------|:--------------:|:--------------:|
| Tên hàm | Cùng | Cùng |
| **Tham số** | **KHÁC** | **Cùng** |
| Class | Thường 1 class | 2 class (base/derived) |
| Kế thừa | Không cần | **CẦN** |
| `virtual` | Không cần | **CẦN** |
| Thời điểm | **Compile time** | **Runtime** |
| Từ khóa | Không có | `override` (C++11) |
| Mục đích | Xử lý nhiều kiểu | Thay đổi hành vi |

### ⚠️ Lỗi âm thầm phổ biến nhất
```cpp
class Base { virtual void f(int x) {} };
class Derived : public Base {
    void f(double x) {}  // OVERLOAD, không phải OVERRIDE!
    // → Không có lỗi, nhưng logic sai!
    // Dùng 'override' để bắt lỗi này
};
```


In [ ]:
%%writefile x4_compare.cpp
#include <iostream>
#include <string>
using namespace std;

// ─── OVERLOADING: cung class, khac tham so ───────────────────────────────────
class MayTinh {
public:
    int    tinh(int a, int b)       { return a + b; }
    double tinh(double a, double b) { return a + b; }
    string tinh(string a, string b) { return a + b; }  // noi chuoi
    int    tinh(int a, int b, int c){ return a + b + c; }
};

// ─── OVERRIDING: khac class, cung signature ───────────────────────────────────
class Hinh {
public:
    virtual string ten()  const { return "Hinh tong quat"; }
    virtual void   info() const { cout << "Day la: " << ten() << "\n"; }
    virtual ~Hinh() {}
};

class HinhTron : public Hinh {
public:
    string ten()  const override { return "Hinh Tron"; }
    // info() KHONG override → dung cua Hinh
    // Trong Hinh::info(), ten() goi → HinhTron::ten() (virtual!)
};

class HinhChuNhat : public Hinh {
public:
    string ten()  const override { return "Hinh Chu Nhat"; }
    void   info() const override {   // override info() cung
        cout << "[HCN] Dai x rong\n";
    }
};

int main() {
    cout << "╔═════════════════════╗\n";
    cout << "║    OVERLOADING      ║\n";
    cout << "╚═════════════════════╝\n";
    MayTinh mc;
    cout << "tinh(1,2)        = " << mc.tinh(1, 2)             << "\n";
    cout << "tinh(1.5,2.3)    = " << mc.tinh(1.5, 2.3)         << "\n";
    cout << "tinh(\"Xin\",\" chao\") = " << mc.tinh(string("Xin"), string(" chao")) << "\n";
    cout << "tinh(1,2,3)      = " << mc.tinh(1, 2, 3)          << "\n";

    cout << "\n╔═════════════════════╗\n";
    cout << "║    OVERRIDING       ║\n";
    cout << "╚═════════════════════╝\n";
    Hinh*       h   = new Hinh();
    HinhTron*   ht  = new HinhTron();
    HinhChuNhat* hcn = new HinhChuNhat();

    h->info();    // "Hinh tong quat"
    ht->info();   // ten() bi override → "Hinh Tron"
    hcn->info();  // info() bi override

    cout << "\n╔══════════════════════════════════╗\n";
    cout << "║ Qua Base pointer — Runtime call  ║\n";
    cout << "╚══════════════════════════════════╝\n";
    Hinh* shapes[] = {h, ht, hcn};
    for (auto s : shapes) {
        cout << "  → "; s->info();
    }
    cout << "(Runtime quyet dinh goi phien ban nao!)\n";

    delete h; delete ht; delete hcn;
    return 0;
}


In [ ]:
!g++ -std=c++17 -o x4_out x4_compare.cpp && ./x4_out

## X.5 — Hàm ảo · Hàm thuần ảo · Lớp trừu tượng

### X.5.1 — Virtual Function (Hàm ảo)
```cpp
virtual void method() { /* default implementation */ }
```
- Có implementation mặc định
- Derived **có thể** override hoặc không
- Gọi qua base pointer → vtable lookup tìm đúng phiên bản

### X.5.2 — Pure Virtual Function (Hàm thuần ảo)
```cpp
virtual void method() = 0;   // = 0 → pure virtual
```
- **Không có implementation** trong base
- Derived **PHẢI** implement, nếu không → cũng là abstract class
- Class có ít nhất 1 pure virtual → **Abstract Class**

### X.5.3 — Abstract Class (Lớp trừu tượng)
```cpp
class HinhHoc {              // Abstract class
    virtual double dien_tich() = 0;   // pure virtual
    virtual double chu_vi()    = 0;   // pure virtual
};

HinhHoc h;                   // ERROR! Cannot instantiate abstract class
HinhHoc* p = new HinhTron(); // OK: pointer to concrete derived
```

### 💡 Virtual vs Pure Virtual — Khi nào dùng cái nào?
```
virtual f() { default }  →  Derived có thể override, hoặc dùng default
virtual f() = 0          →  Derived BẮT BUỘC phải implement
```

### 🎯 Template Method Pattern — design pattern kinh điển
```cpp
class Base {
    virtual void step1() = 0;      // pure virtual
    virtual void step2() = 0;      // pure virtual
    void run() {                   // non-virtual template
        step1(); step2();          // gọi các bước qua virtual
    }
};
```


In [ ]:
%%writefile x5_virtual.cpp
#include <iostream>
#include <cmath>
#include <vector>
#include <iomanip>
using namespace std;

// ─── ABSTRACT BASE CLASS ─────────────────────────────────────────────────────
class HinhHoc {
public:
    // Pure virtual — Derived BUOC PHAI implement
    virtual double dien_tich() const = 0;
    virtual double chu_vi()    const = 0;
    virtual string ten()       const = 0;

    // Regular virtual — co default, Derived co the override
    virtual void in_thong_tin() const {
        cout << fixed << setprecision(2);
        cout << "┌─ " << ten() << "\n";
        cout << "│  Dien tich: " << dien_tich() << "\n";
        cout << "└─ Chu vi:    " << chu_vi()    << "\n";
    }

    virtual ~HinhHoc() {}
};

// ─── CONCRETE CLASSES ─────────────────────────────────────────────────────────
class HinhTron : public HinhHoc {
    double r;
public:
    HinhTron(double r) : r(r) {}
    double dien_tich() const override { return M_PI * r * r; }
    double chu_vi()    const override { return 2 * M_PI * r; }
    string ten()       const override { return "Hinh Tron (r=" + to_string((int)r) + ")"; }
};

class HinhChuNhat : public HinhHoc {
    double d, r;
public:
    HinhChuNhat(double d, double r) : d(d), r(r) {}
    double dien_tich() const override { return d * r; }
    double chu_vi()    const override { return 2 * (d + r); }
    string ten()       const override { return "HCN (" + to_string((int)d) + "x" + to_string((int)r) + ")"; }
    void in_thong_tin() const override {   // override in_thong_tin
        HinhHoc::in_thong_tin();           // goi base
        cout << "  (Ti le: " << fixed << setprecision(2) << d/r << ")\n";
    }
};

class HinhTamGiac : public HinhHoc {
    double a, b, c;
public:
    HinhTamGiac(double a, double b, double c) : a(a), b(b), c(c) {}
    double dien_tich() const override {
        double s = (a+b+c)/2;
        return sqrt(s*(s-a)*(s-b)*(s-c));  // Heron
    }
    double chu_vi() const override { return a + b + c; }
    string ten()    const override { return "Tam Giac"; }
};

int main() {
    // HinhHoc h;  // ERROR: abstract class!
    cout << "(HinhHoc la abstract → khong tao object duoc!)\n\n";

    vector<HinhHoc*> shapes = {
        new HinhTron(5),
        new HinhChuNhat(3, 4),
        new HinhTamGiac(3, 4, 5)
    };

    cout << "=== Thong tin cac hinh ===\n";
    for (auto s : shapes) s->in_thong_tin();

    cout << "\n=== Tinh toan polymorphic ===\n";
    double tong_dt = 0, tong_cv = 0;
    for (auto s : shapes) {
        tong_dt += s->dien_tich();
        tong_cv += s->chu_vi();
    }
    cout << fixed << setprecision(2);
    cout << "Tong dien tich: " << tong_dt << "\n";
    cout << "Tong chu vi:    " << tong_cv << "\n";

    cout << "\n=== Hinh lon nhat ===\n";
    HinhHoc* lon_nhat = shapes[0];
    for (auto s : shapes)
        if (s->dien_tich() > lon_nhat->dien_tich()) lon_nhat = s;
    cout << "=> " << lon_nhat->ten()
         << " (S = " << lon_nhat->dien_tich() << ")\n";

    for (auto s : shapes) delete s;
    return 0;
}


In [ ]:
!g++ -std=c++17 -o x5_out x5_virtual.cpp && ./x5_out

## X.6 — Bảng hàm ảo (vtable) — Cơ chế bên trong C++

### 🧠 vtable là gì?

Khi class có virtual function, compiler tự động:
1. Tạo **vtable** (bảng hàm ảo) cho mỗi class — **shared** bởi tất cả objects
2. Thêm **vptr** (virtual pointer — 8 bytes) vào đầu mỗi object

```
     DongVat object                  DongVat vtable
     ┌────────────────┐              ┌─────────────────────┐
     │  vptr      ────┼─────────────►│ DongVat::keu()      │
     │  ten           │              │ DongVat::tiep_theo() │
     │  tuoi          │              └─────────────────────┘
     └────────────────┘

     Cho object                      Cho vtable
     ┌────────────────┐              ┌─────────────────────┐
     │  vptr      ────┼─────────────►│ Cho::keu()   ← override!
     │  ten (base)    │              │ DongVat::tiep_theo() ← inherited
     │  tuoi (base)   │              └─────────────────────┘
     │  giong         │
     └────────────────┘
```

### 🔧 Cơ chế dispatch khi gọi `p->keu()`

```
1. CPU đọc vptr từ object (offset 0)
2. vptr → địa chỉ vtable của class thực sự
3. vtable[keu] → địa chỉ hàm keu() đúng
4. CPU nhảy đến địa chỉ và thực thi
→ Overhead: 2 lần đọc memory thêm (thường bỏ qua được)
```

### 📊 Chi phí virtual function

| | Không có virtual | Có virtual |
|--|:----------------:|:----------:|
| Mỗi object | 0 bytes thêm | **+8 bytes** (vptr) |
| Mỗi lần gọi | Trực tiếp | 2 lần deref thêm |
| Tổng overhead | Nhỏ, thường bỏ qua |

### ⚠️ Non-virtual function — KHÔNG qua vtable!
```cpp
p->non_virtual();  // Compile time → địa chỉ cố định → STATIC binding
```


In [ ]:
%%writefile x6_vtable.cpp
#include <iostream>
using namespace std;

class Base {
public:
    virtual void f() { cout << "Base::f()  [virtual]\n"; }
    virtual void g() { cout << "Base::g()  [virtual]\n"; }
    void         h() { cout << "Base::h()  [NON-virtual]\n"; }
    virtual ~Base() {}
};

class Derived : public Base {
public:
    void f() override { cout << "Derived::f()  [override]\n"; }
    // g() KHONG override → dung cua Base
    void h()          { cout << "Derived::h()  [shadows Base::h]\n"; }
};

// ─── Demo memory overhead ─────────────────────────────────────────────────────
struct NoVirtual   { int x; };
struct WithVirtual { int x; virtual void f() {} };

int main() {
    cout << "=== Memory overhead cua vtable ===\n";
    cout << "sizeof(NoVirtual)   = " << sizeof(NoVirtual)   << " bytes\n";
    cout << "sizeof(WithVirtual) = " << sizeof(WithVirtual) << " bytes\n";
    cout << "(Chenh lech = size cua vptr = " << sizeof(void*) << " bytes)\n\n";

    Base*    pb  = new Base();
    Derived* pd  = new Derived();
    Base*    pbd = new Derived();  // Base pointer → Derived object!

    cout << "── pb  (Base*→Base) ──\n";
    pb->f();  pb->g();  pb->h();

    cout << "\n── pd  (Derived*→Derived) ──\n";
    pd->f();  pd->g();  pd->h();

    cout << "\n── pbd (Base*→Derived) — vtable dieu phoi ──\n";
    pbd->f();  // Derived::f — vtable cua Derived co Derived::f
    pbd->g();  // Base::g   — vtable cua Derived co Base::g (khong override)
    pbd->h();  // Base::h   — h() la NON-virtual → STATIC binding!
    cout << "(f: vtable → Derived::f)\n";
    cout << "(g: vtable → Base::g, Derived khong override)\n";
    cout << "(h: NON-virtual → compile time fix → Base::h)\n";

    delete pb; delete pd; delete pbd;
    return 0;
}


In [ ]:
!g++ -std=c++17 -o x6_out x6_vtable.cpp && ./x6_out

## X.7 — Con trỏ Lớp cơ sở & Object Slicing

### 🧠 Con trỏ Base Class — Sức mạnh của Polymorphism

```cpp
HinhHoc* p = new HinhTron(5);
p->dien_tich();   // gọi HinhTron::dien_tich() — đúng!
```

Nhờ vtable, `p` biết cần gọi đúng phiên bản dù chỉ là `HinhHoc*`.

### 🔥 Object Slicing — Cạm bẫy nguy hiểm!

**Khi copy by value lên base type:**
```cpp
HinhTron ht(5);
HinhHoc  h = ht;    // COPY! Chỉ copy phần HinhHoc — CẮT MẤT phần HinhTron!
h.dien_tich();      // gọi HinhHoc::dien_tich() — SAI!
```

```
Before slicing:                    After slicing:
┌─────────────────────┐            ┌──────────────────┐
│ HinhTron object     │  copy →    │ HinhHoc object   │
│   [HinhHoc part]    │            │   [HinhHoc part] │ ← chỉ copy phần này
│   [HinhTron part]   │            └──────────────────┘
│   (ban kinh r=5)    │              (r BỊ MẤT!)
└─────────────────────┘
```

**Giải pháp: Dùng pointer hoặc reference:**
```cpp
HinhHoc*  p = &ht;   // NO SLICING ✓
HinhHoc&  r = ht;    // NO SLICING ✓
```

### 💡 dynamic_cast — Downcast an toàn
```cpp
HinhHoc* p = new HinhTron(5);
HinhTron* ht  = dynamic_cast<HinhTron*>(p);   // thành công → ht != nullptr
HinhChuNhat* hcn = dynamic_cast<HinhChuNhat*>(p); // thất bại → nullptr
```


In [ ]:
%%writefile x7_baseptr.cpp
#include <iostream>
#include <vector>
using namespace std;

class DongVat {
    string ten;
public:
    DongVat(string t) : ten(t) {}
    virtual void    keu()  const { cout << ten << ": ...\n"; }
    virtual string  loai() const { return "DongVat"; }
    string getTen()        const { return ten; }
    virtual ~DongVat() {}
};

class Cho : public DongVat {
public:
    Cho(string t) : DongVat(t) {}
    void keu() const override { cout << getTen() << ": GAU GAU!\n"; }
    string loai() const override { return "Cho"; }
    void an_xuong() const { cout << getTen() << " dang nhai xuong!\n"; }
};

class Meo : public DongVat {
public:
    Meo(string t) : DongVat(t) {}
    void keu() const override { cout << getTen() << ": MEO MEO!\n"; }
    string loai() const override { return "Meo"; }
};

int main() {
    cout << "=== Base Pointer Array — Polymorphism ===\n";
    vector<DongVat*> vuon_thu = {
        new Cho("Rex"), new Meo("Mimi"), new Cho("Buddy"), new Meo("Luna")
    };
    for (auto dv : vuon_thu) dv->keu();  // Goi dung hanh vi tung con!

    cout << "\n=== OBJECT SLICING — canh bao! ===\n";
    Cho cho("Sliced");
    DongVat copy_val = cho;    // COPY → SLICING! Mat phan Cho
    cout << "cho.keu():       "; cho.keu();
    cout << "copy_val.keu():  "; copy_val.keu();  // KHONG goi Cho::keu!
    cout << "=> Slicing: copy_val mat phan Cho!\n";

    cout << "\n=== Pointer/Reference — Khong bi slicing ===\n";
    DongVat* p = &cho;  // pointer — OK
    DongVat& r = cho;   // reference — OK
    cout << "p->keu(): "; p->keu();   // Cho::keu ✓
    cout << "r.keu():  "; r.keu();    // Cho::keu ✓

    cout << "\n=== dynamic_cast — Downcast an toan ===\n";
    DongVat* dv = new Cho("DynCast");

    // Thanh cong
    Cho* c = dynamic_cast<Cho*>(dv);
    if (c) { cout << "Cast Cho* thanh cong → "; c->an_xuong(); }

    // That bai
    Meo* m = dynamic_cast<Meo*>(dv);
    if (!m) cout << "Cast Meo* that bai → nullptr\n";

    for (auto dv2 : vuon_thu) delete dv2;
    delete dv;
    return 0;
}


In [ ]:
!g++ -std=c++17 -o x7_out x7_baseptr.cpp && ./x7_out

## X.8–9 — Liên kết Động & Liên kết Tĩnh

### X.8 — Liên kết Động (Dynamic Binding / Late Binding)

| Đặc điểm | Giá trị |
|---------|---------|
| Thời điểm quyết định | **Runtime** |
| Cơ chế | vtable lookup qua vptr |
| Điều kiện | `virtual` function + pointer/reference |
| Linh hoạt | ✅ Rất cao |
| Hiệu năng | Chậm hơn 1 chút |

```cpp
Base* p = new Derived();
p->virtual_f();   // DYNAMIC: runtime → vtable → Derived::virtual_f
```

### X.9 — Liên kết Tĩnh (Static Binding / Early Binding)

| Đặc điểm | Giá trị |
|---------|---------|
| Thời điểm quyết định | **Compile time** |
| Cơ chế | Compiler hardcode địa chỉ hàm |
| Điều kiện | Non-virtual, hoặc gọi qua object |
| Linh hoạt | Thấp hơn |
| Hiệu năng | ✅ Nhanh hơn |

### 🎯 Quy tắc quyết định

```
→ DYNAMIC Binding khi:
  ✅ Hàm có 'virtual'
  ✅ Gọi qua POINTER hoặc REFERENCE đến base

→ STATIC Binding khi:
  ✅ Hàm KHÔNG có 'virtual'
  ✅ Gọi qua OBJECT trực tiếp (không phải pointer)
  ✅ Gọi tường minh: Base::method()
```

### ⚠️ Điều bất ngờ — non-virtual qua base pointer
```cpp
Derived d;
Base* p = &d;
p->virtual_method();    // DYNAMIC → Derived::virtual_method ✓
p->non_virtual_method();// STATIC  → Base::non_virtual_method (KHÔNG phải Derived!)
```


In [ ]:
%%writefile x89_binding.cpp
#include <iostream>
using namespace std;

class Hinh {
public:
    virtual void ve()    { cout << "Hinh::ve()    [virtual → DYNAMIC]\n"; }
    void         print() { cout << "Hinh::print() [non-virtual → STATIC]\n"; }
    virtual ~Hinh() {}
};

class HinhTron : public Hinh {
public:
    void ve() override   { cout << "HinhTron::ve()    [override]\n"; }
    void print()         { cout << "HinhTron::print() [shadows Hinh::print]\n"; }
};

int main() {
    HinhTron ht;
    Hinh*    p = &ht;   // Base pointer → Derived object
    Hinh&    r = ht;    // Base reference → Derived object

    cout << "════════════════════════════════════════\n";
    cout << "1. Qua Base POINTER (p → HinhTron)\n";
    cout << "════════════════════════════════════════\n";
    p->ve();     // DYNAMIC: runtime → vtable → HinhTron::ve()
    p->print();  // STATIC:  compile → Hinh::print() (non-virtual!)

    cout << "\n════════════════════════════════════════\n";
    cout << "2. Qua Base REFERENCE (r → HinhTron)\n";
    cout << "════════════════════════════════════════\n";
    r.ve();      // DYNAMIC: runtime → HinhTron::ve()
    r.print();   // STATIC:  compile → Hinh::print()

    cout << "\n════════════════════════════════════════\n";
    cout << "3. Qua OBJECT truc tiep (ht)\n";
    cout << "════════════════════════════════════════\n";
    ht.ve();     // Compiler biet kieu HinhTron → truc tiep goi HinhTron::ve
    ht.print();  // → HinhTron::print (shadow)

    cout << "\n════════════════════════════════════════\n";
    cout << "4. Qua Derived POINTER (pt → HinhTron)\n";
    cout << "════════════════════════════════════════\n";
    HinhTron* pt = &ht;
    pt->ve();    // HinhTron::ve
    pt->print(); // HinhTron::print

    cout << "\n════════════════════════════════════════\n";
    cout << "TOM TAT\n";
    cout << "════════════════════════════════════════\n";
    cout << "p->ve()    (Base*, virtual)     → DYNAMIC → HinhTron::ve\n";
    cout << "p->print() (Base*, non-virtual) → STATIC  → Hinh::print\n";
    cout << "ht.ve()    (object, virtual)    → STATIC  → HinhTron::ve\n";
    cout << "ht.print() (object, non-virtual)→ STATIC  → HinhTron::print\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -o x89_out x89_binding.cpp && ./x89_out

---
## ★ BÀI TỔNG HỢP — Hệ thống Quản lý Nhân Viên

### Tích hợp TẤT CẢ kiến thức IX + X

```
NhanVien (Abstract Base)
  ├── tinh_luong() = 0   (pure virtual)
  ├── loai()       = 0   (pure virtual)
  ├── in()               (virtual — template method)
  │
  ├── NVToanThoiGian      (Concrete Derived)
  │     └── NVQuanLy       (Multi-level Derived — kế thừa 3 cấp)
  │
  └── NVBanThoiGian       (Concrete Derived)
```

**Minh họa:**
- ✅ Abstract class + pure virtual functions
- ✅ Multi-level inheritance (3 cấp)
- ✅ Constructor chain (NhanVien → NVToanThoiGian → NVQuanLy)
- ✅ Virtual destructor
- ✅ Method overriding với `override`
- ✅ Gọi `Base::method()` trong override
- ✅ Base pointer array — full polymorphism
- ✅ `dynamic_cast` để phân loại runtime


In [ ]:
%%writefile capstone_nhanvien.cpp
#include <iostream>
#include <vector>
#include <iomanip>
#include <string>
using namespace std;

// ─── ABSTRACT BASE CLASS ─────────────────────────────────────────────────────
class NhanVien {
protected:
    string ten;
    int    ma_nv;
    int    nam_vao;
public:
    NhanVien(string t, int ma, int nam)
        : ten(t), ma_nv(ma), nam_vao(nam) {}

    // Pure virtual — bắt buộc derived implement
    virtual double tinh_luong() const = 0;
    virtual string loai()       const = 0;

    // Regular virtual — có default, Derived có thể override
    virtual void in() const {
        cout << fixed << setprecision(0);
        cout << "┌─[" << loai() << "] " << ten << " (Ma: " << ma_nv << ")\n";
        cout << "│  Nam vao: " << nam_vao
             << "  |  Luong: " << tinh_luong() << " VND\n";
        cout << "└─────────────────────────────\n";
    }

    string getTen() const { return ten; }
    int    getMa()  const { return ma_nv; }
    virtual ~NhanVien() {}   // virtual destructor!
};

// ─── NHÂN VIÊN TOÀN THỜI GIAN ────────────────────────────────────────────────
class NVToanThoiGian : public NhanVien {
    double luong_co_ban;
    double he_so;   // he so khen thuong
public:
    NVToanThoiGian(string t, int ma, int nam, double luong, double hs = 1.0)
        : NhanVien(t, ma, nam), luong_co_ban(luong), he_so(hs) {}

    double tinh_luong() const override { return luong_co_ban * he_so; }
    string loai()       const override { return "Toan thoi gian"; }
    double getLuongCoBan() const { return luong_co_ban; }
};

// ─── NHÂN VIÊN BÁN THỜI GIAN ─────────────────────────────────────────────────
class NVBanThoiGian : public NhanVien {
    double luong_gio;
    int    so_gio;
public:
    NVBanThoiGian(string t, int ma, int nam, double lg, int sg)
        : NhanVien(t, ma, nam), luong_gio(lg), so_gio(sg) {}

    double tinh_luong() const override { return luong_gio * so_gio; }
    string loai()       const override { return "Ban thoi gian"; }
    void in() const override {
        NhanVien::in();   // goi base::in() truoc
        cout << "  (" << so_gio << " gio x " << luong_gio << " VND/gio)\n";
    }
};

// ─── NHÂN VIÊN QUẢN LÝ (3 CẤP) ──────────────────────────────────────────────
class NVQuanLy : public NVToanThoiGian {
    double thuong_quan_ly;
    string phong_ban;
    int    so_cap_duoi;
public:
    NVQuanLy(string t, int ma, int nam, double luong,
             double thuong, string phong, int cap_duoi)
        : NVToanThoiGian(t, ma, nam, luong),
          thuong_quan_ly(thuong), phong_ban(phong), so_cap_duoi(cap_duoi) {}

    double tinh_luong() const override {
        // Goi luong cua NVToanThoiGian + thuong quan ly
        return NVToanThoiGian::tinh_luong() + thuong_quan_ly;
    }
    string loai() const override { return "Quan ly - " + phong_ban; }
    void in() const override {
        NhanVien::in();  // bo qua NVToanThoiGian::in(), dung NhanVien::in()
        cout << "  Phong ban: " << phong_ban
             << "  |  " << so_cap_duoi << " nhan vien cap duoi\n";
        cout << "  (Luong co ban: " << getLuongCoBan()
             << " + Thuong QL: " << thuong_quan_ly << ")\n";
    }
};

// ─── BÁO CÁO POLYMORPHIC ─────────────────────────────────────────────────────
void in_bao_cao(const vector<NhanVien*>& ds) {
    cout << "\n========================================\n";
    cout << "         BAO CAO NHAN SU              \n";
    cout << "========================================\n";
    double tong = 0;
    for (auto nv : ds) {
        nv->in();        // POLYMORPHISM — goi dung phien ban cho tung loai!
        tong += nv->tinh_luong();
    }
    cout << fixed << setprecision(0);
    cout << "\nTong quy luong: " << tong << " VND\n";
    cout << "So nhan vien:   " << ds.size() << "\n";
}

int main() {
    vector<NhanVien*> cty;
    cty.push_back(new NVToanThoiGian("Nguyen An",  1001, 2020, 15000000, 1.2));
    cty.push_back(new NVBanThoiGian ("Tran Binh",  1002, 2022, 50000,   80 ));
    cty.push_back(new NVToanThoiGian("Le Cuong",   1003, 2021, 12000000     ));
    cty.push_back(new NVBanThoiGian ("Pham Dung",  1004, 2023, 60000,   60 ));
    cty.push_back(new NVQuanLy      ("Hoang Minh", 1005, 2018, 20000000,
                                     8000000, "Ky thuat", 5));

    // Polymorphic operation
    in_bao_cao(cty);

    // dynamic_cast phan loai runtime
    cout << "\n=== Phan loai nhan vien ===\n";
    for (auto nv : cty) {
        if      (dynamic_cast<NVQuanLy*>(nv))
            cout << "[QL] " << nv->getTen() << "\n";
        else if (dynamic_cast<NVToanThoiGian*>(nv))
            cout << "[FT] " << nv->getTen() << "\n";
        else if (dynamic_cast<NVBanThoiGian*>(nv))
            cout << "[PT] " << nv->getTen() << "\n";
    }

    for (auto nv : cty) delete nv;  // virtual destructor dam bao sach!
    return 0;
}


In [ ]:
!g++ -std=c++17 -o capstone_out capstone_nhanvien.cpp && ./capstone_out

---
# 📝 TỔNG KẾT — CHEAT SHEET

## Kế thừa (Inheritance)

```cpp
// Cú pháp
class Derived : public Base {
    Derived(args) : Base(base_args), my_member(v) { }
};

// Ba kiểu kế thừa
class D : public    Base { };  // pub→pub, pro→pro   IS-A (phổ biến nhất)
class D : protected Base { };  // pub→pro, pro→pro
class D : private   Base { };  // pub→pri, pro→pri   IMPLEMENTED-IN-TERMS-OF

// Override và Final
void method() override { }          // bắt lỗi compile time
void method() override final { }    // không cho override tiếp
class Leaf final : public Base { }; // không cho derive tiếp

// Thứ tự ctor/dtor
// Ctor: Base → Derived | Dtor: Derived → Base (NGƯỢC LẠI)

// Virtual destructor — LUÔN CẦN khi dùng polymorphism!
virtual ~Base() { }
```

## Đa hình (Polymorphism)

```cpp
// Pure virtual + Abstract class
class Interface {
    virtual void f() = 0;      // pure virtual
    virtual ~Interface() { }   // virtual dtor!
};
// Interface i;  // ERROR: cannot instantiate abstract

// Dynamic binding
Base* p = new Derived();
p->virtual_f();     // DYNAMIC: runtime → vtable → Derived::virtual_f
p->non_virtual_f(); // STATIC:  compile → Base::non_virtual_f

// Object slicing — tránh bằng cách dùng pointer/reference!
Derived d;
Base    b = d;     // SLICING — mat phan Derived
Base*   p = &d;    // OK: no slicing
Base&   r = d;     // OK: no slicing

// dynamic_cast — downcast an toan
Derived* d = dynamic_cast<Derived*>(base_ptr);
if (d) { /* cast thanh cong */ }
```

## Nguyên tắc vàng

```
1. LUON co virtual destructor neu class co virtual functions
2. LUON dung 'override' khi override — bat loi compile time
3. Dung pointer/reference de tranh object slicing
4. Abstract class + pure virtual = interface ro rang
5. public inheritance = IS-A (pho bien nhat ~95%)
6. Dynamic dispatch: virtual + Base pointer/reference
7. Static binding: non-virtual HOAC goi qua object truc tiep
```

## So sánh Dynamic vs Static Binding

```
                DYNAMIC                    STATIC
Thoi diem:      Runtime                   Compile time
Dieu kien:      virtual + pointer/ref     non-virtual HOAC object
Co che:         vtable lookup             address cố định
Toc do:         Cham hon 1 chut           Nhanh hon
```
